# 노트북 7. 컨텍스트 매니지먼트 전략

> Phase 3 — 실전 기법: 챗봇을 똑똑하게 만드는 기술

대화가 길어지면 토큰은 폭발하고 비용은 치솟고 품질은 떨어집니다.
이걸 관리하는 전략이 챗봇의 실전 품질을 결정합니다.

**학습 목표**
- 멀티턴 대화에서 토큰이 폭발적으로 증가하는 원리를 이해한다
- Sliding Window, Token 기반 트리밍, 요약 압축, 하이브리드 전략을 구현할 수 있다
- LangChain의 `trim_messages()` 유틸리티를 활용할 수 있다
- 전략별 토큰 사용량과 정보 보존율의 트레이드오프를 설명할 수 있다
- LangGraph 그래프 노드에서 컨텍스트 관리를 적용할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 문제 정의 + 4가지 전략 + LangGraph 적용 | 읽고 실행 |
| Part 2 — 실습 | 전략별 구현 + 비교 실험 | 직접 작성 |
| Part 3 — 챌린지 | 고객 상담 챗봇 최적 전략 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langchain langgraph

import os
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 문제 정의: 멀티턴 토큰 폭발

노트북 3에서 배운 것처럼, LLM은 매 호출 시 **전체 대화 이력**을 전송합니다.
대화가 길어질수록 입력 토큰이 누적되어 비용과 지연이 급증합니다.

```
턴 1: system + user1                        → 입력 50토큰
턴 2: system + user1 + ai1 + user2           → 입력 150토큰
턴 3: system + user1 + ai1 + user2 + ai2 + user3 → 입력 300토큰
...
턴 20: 전체 이력                              → 입력 5,000+토큰
```

아래 코드는 이 누적 과정을 시뮬레이션합니다.

In [ ]:
# 멀티턴 토큰 누적 시뮬레이션
# 실제 대화 대신, 턴당 평균 토큰 수로 계산
avg_user_tokens = 150    # 사용자 메시지 평균
avg_ai_tokens = 600     # AI 응답 평균
system_tokens = 150      # 시스템 프롬프트

cumulative = []
for turn in range(1, 21):
    # 각 턴에서 전송되는 입력 토큰 = system + 이전 전체 대화 + 현재 user
    input_tokens = system_tokens + (turn - 1) * (avg_user_tokens + avg_ai_tokens) + avg_user_tokens
    cumulative.append(input_tokens)

print(f"{'턴':>4} {'입력 토큰':>10} {'막대'}")
print("-" * 55)
for i, tokens in enumerate(cumulative):
    bar = "#" * (tokens // 100)
    print(f"{i+1:>4} {tokens:>10} {bar}")

print(f"\n1턴 → 20턴: 입력 토큰 {cumulative[0]} → {cumulative[-1]} ({cumulative[-1]/cumulative[0]:.0f}배 증가)")

   턴      입력 토큰 막대
-------------------------------------------------------
   1        300 ###
   2       1050 ##########
   3       1800 ##################
   4       2550 #########################
   5       3300 #################################
   6       4050 ########################################
   7       4800 ################################################
   8       5550 #######################################################
   9       6300 ###############################################################
  10       7050 ######################################################################
  11       7800 ##############################################################################
  12       8550 #####################################################################################
  13       9300 #############################################################################################
  14      10050 ###################################################################

### 비용 누적 문제

토큰이 늘어나면 비용도 함께 증가합니다.
20턴 대화의 **총** 입력 토큰은 단순히 20턴째 토큰이 아니라, 1턴부터 20턴까지의 합계입니다.

In [ ]:
# 20턴 대화의 총 비용 추정 (Flash 기준)
input_price = 0.30 / 1_000_000   # $0.15 per 1M input tokens
output_price = 2.50 / 1_000_000  # $0.60 per 1M output tokens

total_input = sum(cumulative)
total_output = 20 * avg_ai_tokens

input_cost = total_input * input_price
output_cost = total_output * output_price
total_cost = input_cost + output_cost

print(f"20턴 대화 비용 추정 (Gemini 2.5 Flash)")
print(f"  총 입력 토큰: {total_input:,}")
print(f"  총 출력 토큰: {total_output:,}")
print(f"  입력 비용: ${input_cost:.4f}")
print(f"  출력 비용: ${output_cost:.4f}")
print(f"  총 비용: ${total_cost:.4f}")
print(f"\n만약 100명이 각 20턴 대화를 한다면: ${total_cost * 100:.2f}")

20턴 대화 비용 추정 (Gemini 2.5 Flash)
  총 입력 토큰: 148,500
  총 출력 토큰: 12,000
  입력 비용: $0.0445
  출력 비용: $0.0300
  총 비용: $0.0746

만약 100명이 각 20턴 대화를 한다면: $7.46


> 컨텍스트 관리 없이 20턴 대화를 100명이 하면 비용이 빠르게 누적됩니다.
> 컨텍스트 관리 전략을 적용하면 이 비용을 50~80% 절감할 수 있습니다.
> 아래에서 네 가지 전략을 배우고, 각각의 트레이드오프를 이해합니다.

## 1.2 Lost in the Middle 현상

컨텍스트가 길어지면 비용만 문제가 아닙니다.
모델은 **긴 컨텍스트의 중간 부분**에 있는 정보를 잘 활용하지 못하는 경향이 있습니다.
이를 **Lost in the Middle** 현상이라 합니다.

```
[잘 기억] 처음 부분 ... [잘 못 기억] 중간 부분 ... [잘 기억] 마지막 부분
```

이 현상 때문에 "대화 초반에 말한 이름"이나 "중간에 언급한 조건"을 모델이 잊어버리는 일이 발생합니다.

아래 코드는 중간에 핵심 정보를 넣고, 모델이 기억하는지 확인합니다.

In [ ]:
# Lost in the Middle 시연
# 긴 대화 중간에 핵심 정보 삽입
messages = [
    SystemMessage("사용자의 질문에 정확히 답변하세요."),
    HumanMessage("안녕하세요"),
    AIMessage("안녕하세요! 무엇을 도와드릴까요?"),
]

# 무관한 대화 5턴 추가
filler_topics = ["날씨", "음식", "운동", "독서", "여행"]
for topic in filler_topics:
    messages.append(HumanMessage(f"{topic}에 대해 한 줄로 알려줘"))
    messages.append(AIMessage(f"{topic}은 일상에서 중요한 요소입니다. 적절히 활용하면 삶의 질이 높아집니다."))

# 핵심 정보 (중간에 삽입)
messages.insert(5, HumanMessage("참고로 제 이름은 '김하늘'이고, 비밀번호 복구 코드는 'ALPHA-7942'입니다."))
messages.insert(6, AIMessage("네, 기록했습니다."))

# 무관한 대화 5턴 더 추가
for topic in ["영화", "음악", "게임", "요리", "기술"]:
    messages.append(HumanMessage(f"{topic}에 대해 한 줄로 알려줘"))
    messages.append(AIMessage(f"{topic}은 현대 문화의 중요한 부분입니다."))

# 중간 정보 질문
messages.append(HumanMessage("제 이름이 뭐였죠? 그리고 복구 코드는요?"))

response = llm.invoke(messages)
print(f"총 메시지 수: {len(messages)}")
print(f"\n모델 응답: {response.content}")

총 메시지 수: 26

모델 응답: 사용자님의 이름은 **김하늘**이고, 비밀번호 복구 코드는 **ALPHA-7942**입니다.


> 핵심 문제 정리
> - **비용 폭발**: 턴이 늘수록 입력 토큰이 선형 증가 → 총 비용은 이차(quadratic) 증가
> - **품질 저하**: Lost in the Middle로 중간 정보 누락 위험
> - **지연 증가**: 입력 토큰이 많을수록 TTFT(첫 토큰 도착 시간)도 증가
>
> 이 세 가지 문제를 해결하는 것이 **컨텍스트 매니지먼트**입니다.

## 1.3 전략 1: Sliding Window

가장 간단한 전략입니다.
**최근 N개의 메시지만 유지**하고, 오래된 메시지를 삭제합니다.

```
전체 이력: [sys, u1, a1, u2, a2, u3, a3, u4, a4, u5]
Window=4:  [sys, ───────────────────────── u4, a4, u5]  ← 최근 4개만
```

**장점**: 구현이 간단하고 토큰 사용량이 예측 가능
**단점**: 초기 대화의 중요한 맥락(이름, 목적 등)이 사라질 수 있음

아래 코드는 수동으로 Sliding Window를 구현합니다.

In [ ]:
# 수동 Sliding Window 구현
def sliding_window(messages, window_size, keep_system=True):
    """최근 window_size개의 메시지만 유지. system 메시지는 선택적으로 보존."""
    if keep_system and isinstance(messages[0], SystemMessage):
        system = [messages[0]]
        rest = messages[1:]
    else:
        system = []
        rest = messages

    recent = rest[-window_size:]
    return system + recent

# 테스트: 10개 메시지에서 최근 4개만
sample = [
    SystemMessage("당신은 AI 비서입니다."),
    HumanMessage("제 이름은 김철수입니다."),
    AIMessage("안녕하세요 김철수님!"),
    HumanMessage("오늘 날씨가 좋죠?"),
    AIMessage("네, 화창한 날씨입니다."),
    HumanMessage("점심 추천해줘"),
    AIMessage("비빔밥은 어떠세요?"),
    HumanMessage("좋아요. 내 이름이 뭐였죠?"),
]

trimmed = sliding_window(sample, window_size=4)

print(f"원본: {len(sample)}개 메시지")
print(f"트리밍: {len(trimmed)}개 메시지")
print()
for msg in trimmed:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

print("\n(이름 정보가 포함된 초기 메시지가 사라졌습니다)")

원본: 8개 메시지
트리밍: 5개 메시지

  [SystemMessage] 당신은 AI 비서입니다.
  [    AIMessage] 네, 화창한 날씨입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?

(이름 정보가 포함된 초기 메시지가 사라졌습니다)


> Sliding Window 설계 시 고려사항
> - Window 크기가 작으면 토큰 절약은 크지만 맥락 손실이 큼
> - Window 크기가 크면 맥락 보존은 좋지만 토큰 절약 효과가 적음
> - 실무에서는 4-8 메시지(2~4턴)를 Window로 많이 사용합니다
> - 시스템 메시지는 항상 보존하는 것이 안전합니다 (모델의 역할과 제약을 유지)

In [ ]:
# Window 크기에 따른 정보 보존 확인
for ws in [2, 4, 6, len(sample) - 1]:
    trimmed = sliding_window(sample, window_size=ws)
    has_name = any("김철수" in m.content for m in trimmed)
    print(f"Window={ws}: {len(trimmed)}개 메시지 | 이름 정보 보존: {has_name}")

Window=2: 3개 메시지 | 이름 정보 보존: False
Window=4: 5개 메시지 | 이름 정보 보존: False
Window=6: 7개 메시지 | 이름 정보 보존: True
Window=7: 8개 메시지 | 이름 정보 보존: True


## 1.4 LangChain trim_messages()

LangChain은 `trim_messages()` 유틸리티를 제공합니다.
수동 구현보다 더 세밀한 제어가 가능합니다.

```python
from langchain_core.messages import trim_messages

trimmed = trim_messages(
    messages,
    max_tokens=6,          # 최대 토큰(또는 메시지) 수
    token_counter=len,     # 카운터 함수 (len = 메시지 개수)
    strategy="last",       # "last" = 최근 유지, "first" = 처음 유지
    include_system=True,   # 시스템 메시지 항상 보존
    start_on="human",      # 결과가 human 메시지로 시작하도록 보장
)
```

아래 코드는 `trim_messages`의 기본 사용법을 보여줍니다.

참고

`start_on="human"`은 **LLM API의 메시지 규칙**을 지키기 위해 사용합니다.

대부분의 LLM API(OpenAI, Anthropic 등)는 **messages 배열이 반드시 `human`(user) 메시지로 시작**해야 합니다. 만약 `ai`(assistant) 메시지로 시작하면 API가 에러를 반환합니다.

`strategy="last"`로 최근 메시지만 남기면, 잘리는 위치에 따라 **ai 메시지가 맨 앞에 올 수 있습니다**:

```
# 원본
[system, human, ai, human, ai, human, ai]

# trim 후 (start_on 없이) → ai가 맨 앞에 올 수 있음
[system, ai, human, ai]  ← API 에러 발생

# start_on="human" 적용 → 해당 ai 메시지도 제거
[system, human, ai]  ← 정상
```

즉, trim 결과에서 **system 다음 첫 번째 메시지가 반드시 human이 되도록** 앞쪽의 ai 메시지를 추가로 잘라주는 안전장치입니다.

In [ ]:
from langchain_core.messages import trim_messages

# trim_messages 기본 사용 — 메시지 개수 기준
# token_counter=len → 메시지 수를 "토큰"으로 간주
trimmed = trim_messages(
    sample,
    max_tokens=5,         # 최대 5개 메시지
    token_counter=len,
    strategy="last",
    include_system=True,  # system 메시지 항상 보존
)

print(f"원본: {len(sample)}개 → 트리밍: {len(trimmed)}개")
print()
for msg in trimmed:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

원본: 8개 → 트리밍: 5개

  [SystemMessage] 당신은 AI 비서입니다.
  [    AIMessage] 네, 화창한 날씨입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?


### include_system과 start_on 옵션

- `include_system=True`: 시스템 메시지를 항상 결과에 포함 (토큰 계산에서 제외)
- `start_on="human"`: 결과가 반드시 HumanMessage로 시작하도록 보장

`start_on`이 중요한 이유: AI 응답으로 시작하는 대화를 모델에 보내면 혼란을 줄 수 있습니다.

In [ ]:
# start_on 효과 비교
print("=== start_on 없이 ===")
trimmed_no_start = trim_messages(
    sample,
    max_tokens=5,
    token_counter=len,
    strategy="last",
    include_system=True,
)
for msg in trimmed_no_start:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:50]}")

print("\n=== start_on='human' ===")
trimmed_with_start = trim_messages(
    sample,
    max_tokens=5,
    token_counter=len,
    strategy="last",
    include_system=True,
    start_on="human",
)
for msg in trimmed_with_start:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:50]}")

=== start_on 없이 ===
  [SystemMessage] 당신은 AI 비서입니다.
  [    AIMessage] 네, 화창한 날씨입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?

=== start_on='human' ===
  [SystemMessage] 당신은 AI 비서입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?


> trim_messages 주요 파라미터
>
> | 파라미터 | 설명 | 기본값 |
> |---------|------|-------|
> | max_tokens | 최대 허용 토큰(또는 메시지) 수 | 필수 |
> | token_counter | 토큰 계산 방법 (`len`, `llm`, 커스텀 함수) | 필수 |
> | strategy | `"last"` (최근 유지) 또는 `"first"` (처음 유지) | `"last"` |
> | include_system | 시스템 메시지 항상 포함 | `False` |
> | start_on | 결과가 시작해야 할 메시지 타입 (`"human"`) | `None` |
> | allow_partial | 메시지를 잘라서라도 토큰 한도에 맞출지 | `False` |

## 1.5 전략 2: Token 기반 트리밍

Sliding Window는 메시지 개수로 자르지만, 실제 비용은 **토큰 수**에 비례합니다.
짧은 메시지 10개와 긴 메시지 2개는 토큰 수가 크게 다릅니다.

Token 기반 트리밍은 `token_counter`에 실제 토큰 카운터를 전달하여
정확한 토큰 예산 내에서 메시지를 유지합니다.

아래 코드는 LLM 모델을 token_counter로 사용합니다.

In [ ]:
# Token 기반 트리밍 — llm을 token_counter로 사용
# ChatGoogleGenerativeAI는 내부적으로 토큰을 계산
trimmed_by_token = trim_messages(
    sample,
    max_tokens=64,         # 64토큰 이내
    token_counter=llm,      # 실제 토큰 카운터
    strategy="last",
    include_system=True,
    start_on="human",
)

print(f"원본: {len(sample)}개 메시지")
print(f"100토큰 제한 후: {len(trimmed_by_token)}개 메시지")
print()
for msg in trimmed_by_token:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

원본: 8개 메시지
100토큰 제한 후: 6개 메시지

  [SystemMessage] 당신은 AI 비서입니다.
  [ HumanMessage] 오늘 날씨가 좋죠?
  [    AIMessage] 네, 화창한 날씨입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?


참고: 커스텀 함수 만들기 — 기본 패턴

```python
def custom_token_counter(messages: list) -> int:
    """메시지 리스트를 받아 총 토큰 수(int)를 반환"""
    text = "\n".join(m.content for m in messages)
    # 여기서 원하는 토큰 계산 로직 사용
    return 계산된_토큰_수
```
규칙:

입력: list[BaseMessage] (SystemMessage, HumanMessage, AIMessage 등)
출력: int (총 토큰 수)

각 메시지의 텍스트는 m.content로 접근

In [ ]:
# 커스텀 토큰 카운터 — google-genai count_tokens 활용
def custom_token_counter(messages) -> int:
    """google-genai의 count_tokens로 정확한 토큰 수를 계산합니다."""
    text = "\n".join(m.content for m in messages)
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

# trim_messages에 커스텀 카운터 사용
trimmed_custom = trim_messages(
    sample,
    max_tokens=64,
    token_counter=custom_token_counter,
    strategy="last",
    include_system=True,
    start_on="human",
)

print(f"커스텀 카운터(32토큰 제한) 결과: {len(trimmed_custom)}개 메시지")
for msg in trimmed_custom:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

커스텀 카운터(32토큰 제한) 결과: 6개 메시지
  [SystemMessage] 당신은 AI 비서입니다.
  [ HumanMessage] 오늘 날씨가 좋죠?
  [    AIMessage] 네, 화창한 날씨입니다.
  [ HumanMessage] 점심 추천해줘
  [    AIMessage] 비빔밥은 어떠세요?
  [ HumanMessage] 좋아요. 내 이름이 뭐였죠?


### 턴 수 vs 토큰 수 비교

같은 대화도 메시지 길이에 따라 토큰 수가 다릅니다.
아래 코드는 메시지 길이가 불균일한 대화에서 두 방식의 차이를 보여줍니다.

In [ ]:
# 메시지 길이가 불균일한 대화
uneven_messages = [
    SystemMessage("당신은 AI 비서입니다."),
    HumanMessage("안녕"),                                                # 짧음
    AIMessage("안녕하세요!"),                                             # 짧음
    HumanMessage("파이썬의 역사, 특징, 장단점, 활용 분야에 대해 자세히 설명해주세요."),  # 김
    AIMessage("파이썬은 1991년 귀도 반 로썸이 만든 프로그래밍 언어입니다. " * 10),      # 매우 김
    HumanMessage("고마워"),                                              # 짧음
]

# 메시지 수 기준: 최근 3개
by_count = trim_messages(uneven_messages, max_tokens=4, token_counter=len,
                         strategy="last", include_system=True)

# 토큰 수 기준: 150토큰
by_token = trim_messages(uneven_messages, max_tokens=128, token_counter=llm,
                         strategy="last", include_system=True, start_on="human")

print("=== 메시지 수 기준 (max=4) ===")
for msg in by_count:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:50]}")

print("\n=== 토큰 수 기준 (max=150) ===")
for msg in by_token:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:50]}")

print("\n(토큰 기준은 긴 메시지를 더 적극적으로 제거합니다)")

=== 메시지 수 기준 (max=4) ===
  [SystemMessage] 당신은 AI 비서입니다.
  [ HumanMessage] 파이썬의 역사, 특징, 장단점, 활용 분야에 대해 자세히 설명해주세요.
  [    AIMessage] 파이썬은 1991년 귀도 반 로썸이 만든 프로그래밍 언어입니다. 파이썬은 1991년 귀도 
  [ HumanMessage] 고마워

=== 토큰 수 기준 (max=150) ===
  [SystemMessage] 당신은 AI 비서입니다.
  [ HumanMessage] 고마워

(토큰 기준은 긴 메시지를 더 적극적으로 제거합니다)


### 요약 프롬프트 설계

요약의 품질은 프롬프트에 크게 의존합니다.

1. 좋은 요약 프롬프트는 **보존해야 할 정보의 종류**를 명시합니다.

- 나쁜 프롬프트: \"대화를 요약해줘\"

- 좋은 프롬프트: \"대화를 요약해주세요. 사용자 이름, 요청 사항, 합의된 내용을 반드시 포함하세요.

2. 도메인에 따라 보존 항목이 달라집니다

-  **고객 상담**: 이름, 주문번호, 문의 내용, 해결 상태
-  **교육**: 학생 이름, 학습 주제, 이해도, 다음 단계
- **기술 지원**: 에러 내용, 시도한 해결 방법, 환경 정보

## 1.6 전략 3: 요약 기반 압축

Sliding Window와 Token 트리밍은 오래된 메시지를 **삭제**합니다.
요약 기반 압축은 오래된 대화를 LLM으로 **요약**하여 핵심 정보를 보존합니다.

```
전략 흐름:
1. 대화가 임계값을 초과하면
2. 오래된 메시지를 LLM으로 요약
3. 요약본을 시스템 프롬프트에 포함
4. 최근 메시지만 유지
```

**장점**: 핵심 정보(이름, 주제, 합의사항)가 보존됨
**단점**: 추가 LLM 호출 비용, 요약 과정에서 세부사항 손실 가능

아래 코드는 요약 함수를 구현합니다.

In [ ]:
# 요약 함수 구현
def summarize_conversation(messages, llm):
    """메시지 리스트를 LLM으로 요약합니다."""
    conversation_text = "\n".join(
        f"{type(m).__name__}: {m.content}" for m in messages
        if not isinstance(m, SystemMessage)
    )

    summary_prompt = (
        "다음 대화를 핵심 정보 위주로 간결하게 요약해주세요.\n"
        "사용자의 이름, 요청 사항, 합의된 내용 등을 반드시 포함하세요.\n\n"
        f"{conversation_text}"
    )

    response = llm.invoke(summary_prompt)
    return response.content

# 테스트
test_messages = [
    HumanMessage("안녕하세요, 저는 이영희입니다."),
    AIMessage("안녕하세요 이영희님!"),
    HumanMessage("파이썬으로 웹 크롤링하는 법을 알려주세요."),
    AIMessage("requests와 BeautifulSoup 라이브러리를 사용하면 됩니다."),
    HumanMessage("로그인이 필요한 사이트도 가능한가요?"),
    AIMessage("네, requests.Session()을 사용하면 로그인 세션을 유지할 수 있습니다."),
]

summary = summarize_conversation(test_messages, llm)
print(f"원본 메시지: {len(test_messages)}개")
print(f"\n요약 결과:\n{summary}")

원본 메시지: 6개

요약 결과:
사용자 이영희님은 파이썬으로 웹 크롤링하는 방법을 요청했으며, AI는 `requests`와 `BeautifulSoup` 라이브러리 사용을 안내했습니다. 로그인이 필요한 사이트의 경우 `requests.Session()`을 사용하여 로그인 세션을 유지할 수 있다고 합의했습니다.


### 요약 적용: 오래된 대화를 요약본으로 교체

요약본을 시스템 프롬프트에 포함하고, 최근 메시지만 유지하는 패턴입니다.

In [ ]:
# 요약 기반 압축 적용
def compress_with_summary(messages, llm, keep_recent=4):
    """오래된 메시지를 요약하고 최근 메시지만 유지합니다."""
    # 시스템 메시지 분리
    system_msg = messages[0] if isinstance(messages[0], SystemMessage) else None
    non_system = messages[1:] if system_msg else messages

    if len(non_system) <= keep_recent:
        return messages  # 충분히 짧으면 그대로

    # 오래된 부분 요약
    old_messages = non_system[:-keep_recent]
    recent_messages = non_system[-keep_recent:]

    summary = summarize_conversation(old_messages, llm)

    # 새 메시지 리스트 구성
    base_instruction = system_msg.content if system_msg else "당신은 AI 비서입니다."
    new_system = SystemMessage(f"{base_instruction}\n\n[이전 대화 요약]\n{summary}")

    return [new_system] + recent_messages

# 테스트: 8개 메시지 → 요약 + 최근 4개
full_conversation = [
    SystemMessage("당신은 친절한 프로그래밍 튜터입니다."),
    *test_messages,
    HumanMessage("Selenium도 사용할 수 있나요?"),
    AIMessage("네, 동적 웹페이지에는 Selenium이 적합합니다."),
]

compressed = compress_with_summary(full_conversation, llm, keep_recent=4)

print(f"원본: {len(full_conversation)}개 → 압축: {len(compressed)}개")
print()
for msg in compressed:
    content = msg.content
    print(f"  [{type(msg).__name__:>13}] {content}")

원본: 9개 → 압축: 5개

  [SystemMessage] 당신은 친절한 프로그래밍 튜터입니다.

[이전 대화 요약]
사용자 이영희님은 파이썬 웹 크롤링 방법을 문의했으며, `requests`와 `BeautifulSoup` 라이브러리를 사용하는 것이 제안되었습니다.
  [ HumanMessage] 로그인이 필요한 사이트도 가능한가요?
  [    AIMessage] 네, requests.Session()을 사용하면 로그인 세션을 유지할 수 있습니다.
  [ HumanMessage] Selenium도 사용할 수 있나요?
  [    AIMessage] 네, 동적 웹페이지에는 Selenium이 적합합니다.


> 요약 전략 주의사항
> - 요약 자체에 LLM 호출 비용이 발생합니다 (요약 입력 토큰 + 요약 출력 토큰)
> - 요약 과정에서 세부 수치, 코드 조각 등이 손실될 수 있습니다
> - 요약 프롬프트에 "보존해야 할 정보"를 명시하면 품질이 개선됩니다
> - 대화가 짧을 때는 요약 비용이 절약 비용보다 클 수 있으므로 임계값을 설정해야 합니다

## 1.7 전략 4: 하이브리드 (요약 + Sliding Window)

실전에서 가장 많이 사용되는 전략입니다.
**요약 + Sliding Window**를 결합하여 각각의 장점을 취합니다.

```
대화 길이 ≤ 임계값 → 그대로 전달 (트리밍 불필요)
대화 길이 > 임계값 → 오래된 부분 요약 + 최근 N턴 유지
```

아래 코드는 하이브리드 전략을 구현합니다.

> 응답 품질 분석
> - **원본**: 모든 정보 보존 (비용이 가장 높음)
> - **Sliding Window**: 최근 대화만 기억. 초기에 언급된 이름/양념 정보를 잊을 가능성 높음
> - **Token 트리밍**: Sliding Window와 유사하지만, 긴 메시지를 더 적극적으로 제거
> - **요약 압축**: 이름과 핵심 요청을 요약에 포함시키므로 정보 보존이 좋음. 단, 세부 수치(양념 양 등)는 손실 가능
>
> 결론: 정보 보존이 중요한 서비스에서는 요약 기반 전략이 유리합니다.

In [ ]:
# 하이브리드 전략 구현
def hybrid_context_management(messages, llm, threshold=8, keep_recent=4):
    """
    하이브리드 컨텍스트 관리.
    - threshold 이하: 그대로 반환
    - threshold 초과: 요약 + 최근 keep_recent개 유지
    """
    if len(messages) <= threshold:
        return messages, "passthrough"

    compressed = compress_with_summary(messages, llm, keep_recent=keep_recent)
    return compressed, "summarized"

# 짧은 대화 → 그대로
short_conv = [
    SystemMessage("당신은 AI 비서입니다."),
    HumanMessage("안녕?"),
    AIMessage("안녕하세요!"),
]

result, action = hybrid_context_management(short_conv, llm)
print(f"짧은 대화 ({len(short_conv)}개): {action} → {len(result)}개 메시지")

# 긴 대화 → 요약
result2, action2 = hybrid_context_management(full_conversation, llm)
print(f"긴 대화 ({len(full_conversation)}개): {action2} → {len(result2)}개 메시지")

짧은 대화 (3개): passthrough → 3개 메시지
긴 대화 (9개): summarized → 5개 메시지


> 하이브리드 임계값 설정 가이드
>
> | 임계값 | keep_recent | 적합한 상황 |
> |-------|------------|----------|
> | 6~8 메시지 | 4 | 일반 챗봇 (빠른 응답 중요) |
> | 12~16 메시지 | 6~8 | 상담/교육 (맥락 중요) |
> | 500~1000 토큰 | 300 토큰 | 토큰 기반 정밀 제어 |

## 1.8 전략별 비교

네 가지 전략의 특성을 비교합니다.

| 전략 | 토큰 절약 | 정보 보존 | 구현 복잡도 | 추가 비용 |
|------|---------|---------|----------|--------|
| Sliding Window | 높음 | 낮음 (초기 정보 손실) | 매우 낮음 | 없음 |
| Token 트리밍 | 높음 (정밀) | 낮음 | 낮음 | 없음 |
| 요약 압축 | 중간 | 높음 (핵심 보존) | 중간 | 요약 LLM 호출 |
| 하이브리드 | 중간~높음 | 높음 | 중간 | 조건부 LLM 호출 |

아래 코드는 동일한 대화에 네 가지 전략을 적용하고 결과를 비교합니다.

In [ ]:
# 전략별 비교 실행
# 비교용 긴 대화 생성

def count_message_tokens(messages, llm=llm):
    """LLM의 내장 토큰 카운터로 정확한 토큰 수 계산"""
    return llm.get_num_tokens_from_messages(messages)

long_conversation = [
    SystemMessage("당신은 요리 전문가 AI입니다."),
    HumanMessage("안녕하세요, 저는 박지민입니다. 초보 요리사예요."),
    AIMessage("안녕하세요 박지민님! 요리를 시작하셨군요. 무엇을 도와드릴까요?"),
    HumanMessage("김치찌개 레시피를 알려주세요."),
    AIMessage("김치찌개는 김치, 돼지고기, 두부, 대파가 필요합니다. 먼저 돼지고기를 볶고 김치를 넣어 볶다가 물을 부어 끓이세요."),
    HumanMessage("양념은 어떻게 하나요?"),
    AIMessage("고추장 1큰술, 고춧가루 1큰술, 다진 마늘 1큰술을 넣으면 됩니다."),
    HumanMessage("된장찌개도 알려주세요."),
    AIMessage("된장찌개는 된장 2큰술, 감자, 애호박, 두부, 청양고추가 필요합니다."),
    HumanMessage("제 이름이 뭐였죠? 그리고 김치찌개에 양념 뭘 넣으라 했죠?"),
]

# 1. 원본
original_tokens = count_message_tokens(long_conversation)

# 2. Sliding Window
sw_result = sliding_window(long_conversation, window_size=4)
sw_tokens = count_message_tokens(sw_result)

# 3. trim_messages (토큰 기반)
tm_result = trim_messages(
    long_conversation, max_tokens=200, token_counter=llm,
    strategy="last", include_system=True, start_on="human",
)
tm_tokens = count_message_tokens(tm_result)

# 4. 요약 압축
summary_result = compress_with_summary(long_conversation, llm, keep_recent=4)
summary_tokens = count_message_tokens(summary_result)

# 비교 표
print(f"{'전략':>15} {'메시지 수':>8} {'토큰 수':>8} {'절약률':>8}")
print("-" * 45)
print(f"{'원본':>15} {len(long_conversation):>8} {original_tokens:>8} {'—':>8}")
print(f"{'Sliding Window':>15} {len(sw_result):>8} {sw_tokens:>8} {(1-sw_tokens/original_tokens)*100:>7.1f}%")
print(f"{'Token 트리밍':>15} {len(tm_result):>8} {tm_tokens:>8} {(1-tm_tokens/original_tokens)*100:>7.1f}%")
print(f"{'요약 압축':>15} {len(summary_result):>8} {summary_tokens:>8} {(1-summary_tokens/original_tokens)*100:>7.1f}%")

             전략    메시지 수     토큰 수      절약률
---------------------------------------------
             원본       10      215        —
 Sliding Window        5      109    49.3%
      Token 트리밍        8      177    17.7%
          요약 압축        5      208     3.3%


> 참고: 위 예시에서 `should_summarize`는 메시지 수를 기준으로 분기합니다.
> 실무에서는 토큰 수 기준으로 분기하는 것이 더 정밀합니다.
> 또한, 요약 결과를 `ChatState.summary`에 저장하므로 이전 요약 위에 새 요약을 누적하는 \"점진적 요약(incremental summarization)\" 패턴도 가능합니다.

In [ ]:
# 각 전략의 응답 품질 비교 — 이름과 양념 정보를 기억하는가?
strategies = {
    "원본": long_conversation,
    "Sliding Window": sw_result,
    "Token 트리밍": tm_result,
    "요약 압축": summary_result,
}

for name, msgs in strategies.items():
    response = llm.invoke(msgs)
    has_name = "박지민" in response.content
    has_recipe = "고추장" in response.content or "고춧가루" in response.content or "마늘" in response.content
    print(f"[{name:>15}] 이름 기억: {'O' if has_name else 'X'} | 양념 기억: {'O' if has_recipe else 'X'}")
    print(f"  응답: {response.content[:100]}")
    print()

[             원본] 이름 기억: O | 양념 기억: O
  응답: 박지민님, 다시 오셨네요!

김치찌개 양념으로는 **고추장 1큰술, 고춧가루 1큰술, 다진 마늘 1큰술**을 넣으라고 말씀드렸습니다.

[ Sliding Window] 이름 기억: X | 양념 기억: O
  응답: 죄송합니다만, 저는 고객님의 이름을 알 수 없습니다. 저는 인공지능이므로 개인 정보를 기억하지 않습니다.

김치찌개 양념으로는 **고추장 1큰술, 고춧가루 1큰술, 다진 마늘 1큰

[      Token 트리밍] 이름 기억: X | 양념 기억: O
  응답: 손님께서는 저에게 이름을 알려주신 적이 없어서 제가 알 수 없습니다.

김치찌개 양념으로는 **고추장 1큰술, 고춧가루 1큰술, 다진 마늘 1큰술**을 넣으시라고 말씀드렸습니다.

[          요약 압축] 이름 기억: O | 양념 기억: O
  응답: 박지민님, 다시 오신 것을 환영합니다!

김치찌개 양념으로는 **고추장 1큰술, 고춧가루 1큰술, 다진 마늘 1큰술**을 넣으시면 됩니다.



> 전략 선택 가이드
> - **빠른 응답, 비용 절감이 최우선** → Sliding Window 또는 Token 트리밍
> - **맥락 보존이 중요** (상담, 교육) → 요약 압축 또는 하이브리드
> - **실전 서비스** → 하이브리드 (짧을 때 패스, 길 때 요약)

## 1.9 LangGraph에서의 컨텍스트 관리

LangGraph의 `StateGraph`에서는 그래프 노드 안에서 컨텍스트 관리를 적용합니다.
`MessagesState`에 저장된 메시지 이력을 LLM에 전달하기 전에 트리밍합니다.

아래 코드는 LangGraph 노드에서 `trim_messages`를 사용하는 패턴입니다.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

# 노드 내부에서 trim_messages 적용
def chatbot_node(state: MessagesState):
    # LLM에 전달하기 전에 트리밍
    trimmed = trim_messages(
        state["messages"],
        max_tokens=300,
        token_counter=llm,
        strategy="last",
        include_system=True,
        start_on="human",
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

# 그래프 구성
graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot_node)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# 대화 실행
config = {"configurable": {"thread_id": "context-demo"}}

# 여러 턴 대화
turns = [
    "안녕, 내 이름은 민수야.",
    "파이썬에 대해 알려줘.",
    "내 이름이 뭐였지?",
]

for user_input in turns:
    result = app.invoke(
        {"messages": [HumanMessage(user_input)]},
        config=config,
    )
    last_msg = result["messages"][-1]
    print(f"User: {user_input}")
    print(f"AI: {last_msg.content[:80]}")
    print()

User: 안녕, 내 이름은 민수야.
AI: 안녕하세요, 민수님! 만나서 반갑습니다.

저는 인공지능 챗봇입니다. 무엇을 도와드릴까요?

User: 파이썬에 대해 알려줘.
AI: 안녕하세요, 민수님! 파이썬에 대해 궁금하시군요. 파이썬은 정말 멋지고 활용도가 높은 프로그래밍 언어입니다. 자세히 설명해 드릴게요.

---


User: 내 이름이 뭐였지?
AI: 저는 사용자님의 이름을 알 수 없습니다. 저는 개인 정보를 저장하거나 기억하지 않기 때문입니다.

혹시 저에게 불러주셨으면 하는 이름이 있으시다



### LangGraph + 요약 노드

더 고급 패턴으로, 요약 전용 노드를 그래프에 추가할 수 있습니다.
대화가 임계값을 초과하면 요약 노드가 실행됩니다.

아래 코드는 조건부 요약 노드가 포함된 그래프를 구성합니다.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

# 요약을 저장할 수 있는 확장 상태
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str

# MessagesState
# class Messagestate(TypedDict):
#     messages: Annotated[list, add_messages]

SUMMARY_THRESHOLD = 6  # 6개 메시지 초과 시 요약

def should_summarize(state: ChatState):
    """메시지 수가 임계값을 초과하면 요약 노드로 분기."""
    if len(state["messages"]) > SUMMARY_THRESHOLD:
        return "summarize"
    return "respond"

def summarize_node(state: ChatState):
    """오래된 메시지를 요약합니다."""
    messages = state["messages"]
    old = messages[:-4]
    summary = summarize_conversation(old, llm)
    # 요약본 저장 + 오래된 메시지 제거 (최근 4개만 유지)
    return {"summary": summary, "messages": messages[-4:]}

def respond_node(state: ChatState):
    """요약이 있으면 시스템 프롬프트에 포함하여 응답."""
    messages = state["messages"]
    summary = state.get("summary", "")

    if summary:
        system = SystemMessage(f"당신은 AI 비서입니다.\n\n[이전 대화 요약] {summary}")
        to_send = [system] + messages
    else:
        to_send = messages

    response = llm.invoke(to_send)
    return {"messages": [response]}

# 그래프 구성
graph2 = StateGraph(ChatState)

graph2.add_node("summarize", summarize_node)
graph2.add_node("respond", respond_node)

graph2.add_conditional_edges(START, should_summarize)
graph2.add_edge("summarize", "respond")
graph2.add_edge("respond", END)

app2 = graph2.compile(checkpointer=MemorySaver())

# 테스트
config2 = {"configurable": {"thread_id": "summary-demo"}}
for msg in ["안녕, 나는 수진이야", "날씨 알려줘", "음식 추천해줘", "내 이름 기억해?"]:
    result = app2.invoke(
        {"messages": [HumanMessage(msg)]},
        config=config2,
    )
    ai_msg = result["messages"][-1]
    summary_status = "있음" if result.get("summary") else "없음"
    print(f"User: {msg}")
    print(f"AI: {ai_msg.content[:80]}")
    print(f"(요약: {summary_status}, 메시지 수: {len(result['messages'])})")
    print()

User: 안녕, 나는 수진이야
AI: 안녕하세요 수진님! 만나서 반갑습니다. 저는 OpenAI에서 훈련한 대규모 언어 모델입니다. 무엇을 도와드릴까요?
(요약: 없음, 메시지 수: 2)

User: 날씨 알려줘
AI: 날씨를 알려드릴 수 있습니다! 그런데 어느 지역의 날씨가 궁금하신가요?

예를 들어, '서울 날씨 알려줘' 또는 '부산 내일 날씨는 어때?'처럼
(요약: 없음, 메시지 수: 4)

User: 음식 추천해줘
AI: 네, 음식 추천해 드릴 수 있습니다! 그런데 어떤 종류의 음식을 좋아하시는지, 어떤 상황이신지 조금 더 알려주시면 더 만족스러운 추천을 해드릴 
(요약: 없음, 메시지 수: 6)

User: 내 이름 기억해?
AI: 네, 수진님! 당연히 기억하고 있습니다. 😊

처음에 '안녕, 나는 수진이야'라고 말씀해주셨을 때부터 기억하고 있었답니다!
(요약: 있음, 메시지 수: 8)



> LangGraph 컨텍스트 관리 팁
> - `trim_messages`를 노드 내부에서 호출하면, 상태에는 전체 이력이 유지되면서 LLM에는 트리밍된 메시지만 전달됩니다
> - 요약 노드를 별도로 두면 요약 타이밍과 전략을 유연하게 제어할 수 있습니다
> - `MemorySaver`(메모리)나 `SqliteSaver`(영속)를 통해 대화 이력 자체는 보존하면서, LLM 호출 시에만 트리밍을 적용하는 패턴이 일반적입니다

---

### 생각해보기

1. Sliding Window에서 시스템 메시지를 항상 보존하는 이유는 무엇일까요? 시스템 메시지를 잘라도 되는 경우가 있을까요?
2. 요약 기반 압축에서 "요약의 품질"은 어떻게 보장할 수 있을까요? 요약이 잘못되면 이후 대화에 어떤 영향을 미칠까요?
3. 토큰 예산을 500으로 설정한 챗봇과 2000으로 설정한 챗봇은 사용자 경험에서 어떤 차이가 있을까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 7-1: Sliding Window 구현

주어진 대화에서 최근 N개 메시지만 유지하는 Sliding Window를 구현하세요.

**요구사항**
1. 시스템 메시지는 항상 보존
2. 나머지 메시지 중 최근 `window_size`개만 유지
3. `window_size=4`로 테스트
4. 트리밍 전/후 메시지 수와 내용을 출력

In [ ]:
# TODO: Sliding Window 구현
practice_messages = [
    SystemMessage("당신은 여행 가이드입니다."),
    HumanMessage("안녕, 나는 지훈이야."),
    AIMessage("안녕하세요 지훈님! 어디로 여행을 계획하고 계신가요?"),
    HumanMessage("일본 여행을 가고 싶어."),
    AIMessage("일본은 도쿄, 오사카, 교토가 인기 있습니다."),
    HumanMessage("도쿄에서 꼭 가봐야 할 곳은?"),
    AIMessage("시부야, 아사쿠사, 아키하바라를 추천합니다."),
    HumanMessage("내 이름이 뭐였지?"),
]

# ---------- 정답 ----------
def my_sliding_window(messages, window_size, keep_system=True):
    if keep_system and isinstance(messages[0], SystemMessage):
        system = [messages[0]]
        rest = messages[1:]
    else:
        system = []
        rest = messages
    return system + rest[-window_size:]

trimmed = my_sliding_window(practice_messages, window_size=4)
print(f"원본: {len(practice_messages)}개 → 트리밍: {len(trimmed)}개")
for msg in trimmed:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

print("TODO를 완성해주세요")

### 실습 7-2: trim_messages 활용

LangChain의 `trim_messages()`를 사용하여 메시지를 트리밍하세요.

**요구사항**
1. `practice_messages`에 `trim_messages` 적용
2. `max_tokens=5`, `token_counter=len`, `strategy="last"`
3. `include_system=True`, `start_on="human"` 설정
4. 결과를 출력하고 시스템 메시지가 보존되었는지 확인

In [ ]:
from langchain_core.messages import trim_messages

# TODO: trim_messages 활용

# ---------- 정답 ----------
trimmed = trim_messages(
    practice_messages,
    max_tokens=5,
    token_counter=len,
    strategy="last",
    include_system=True,
    start_on="human",
)

print(f"원본: {len(practice_messages)}개 → 트리밍: {len(trimmed)}개")
print(f"시스템 메시지 보존: {isinstance(trimmed[0], SystemMessage)}")
print(f"Human으로 시작: {isinstance(trimmed[1], HumanMessage)}")
print()
for msg in trimmed:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

print("TODO를 완성해주세요")

### 실습 7-3: Token 기반 트리밍

`trim_messages`에 실제 토큰 카운터(`llm`)를 사용하여 토큰 예산 내에서 트리밍하세요.

**요구사항**
1. `token_counter=llm` 사용
2. `max_tokens=100`으로 설정
3. `include_system=True`, `start_on="human"`
4. 트리밍 전/후 메시지 수를 출력

In [ ]:
# TODO: Token 기반 트리밍

# ---------- 정답 ----------
trimmed_token = trim_messages(
    practice_messages,
    max_tokens=100,
    token_counter=llm,
    strategy="last",
    include_system=True,
    start_on="human",
)

print(f"원본: {len(practice_messages)}개 → 토큰 트리밍: {len(trimmed_token)}개")
print()
for msg in trimmed_token:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:60]}")

print("TODO를 완성해주세요")

### 실습 7-4: LLM 요약 구현

대화의 오래된 부분을 LLM으로 요약하는 함수를 구현하세요.

**요구사항**
1. `practice_messages`에서 처음 4개 메시지(시스템 제외)를 요약
2. 요약 프롬프트에 "사용자 이름과 핵심 요청 사항을 포함"하도록 지시
3. 요약 결과를 출력

In [ ]:
# TODO: LLM 요약 구현

# ---------- 정답 ----------
def my_summarize(messages, llm):
    text = "\n".join(
        f"{type(m).__name__}: {m.content}" for m in messages
        if not isinstance(m, SystemMessage)
    )
    prompt = (
        "다음 대화를 간결하게 요약해주세요. "
        "사용자 이름과 핵심 요청 사항을 반드시 포함하세요.\n\n"
        f"{text}"
    )
    return llm.invoke(prompt).content

# 처음 4개 메시지 요약 (시스템 제외)
old_messages = practice_messages[1:5]
summary = my_summarize(old_messages, llm)
print(f"요약 대상: {len(old_messages)}개 메시지")
print(f"\n요약 결과:\n{summary}")

print("TODO를 완성해주세요")

### 실습 7-5: 하이브리드 전략 구현

요약 + Sliding Window를 결합한 하이브리드 전략을 구현하세요.

**요구사항**
1. 메시지 수가 6개 초과 시 요약 실행
2. 최근 4개 메시지는 유지
3. 요약본을 시스템 프롬프트에 포함
4. `practice_messages`에 적용하여 결과 출력

In [ ]:
# TODO: 하이브리드 전략 구현

# ---------- 정답 ----------
def my_hybrid(messages, llm, threshold=6, keep_recent=4):
    if len(messages) <= threshold:
        return messages

    system_msg = messages[0] if isinstance(messages[0], SystemMessage) else None
    non_system = messages[1:] if system_msg else messages

    old = non_system[:-keep_recent]
    recent = non_system[-keep_recent:]

    summary = my_summarize(old, llm)
    base = system_msg.content if system_msg else "당신은 AI 비서입니다."
    new_system = SystemMessage(f"{base}\n\n[이전 대화 요약] {summary}")

    return [new_system] + recent

result = my_hybrid(practice_messages, llm)
print(f"원본: {len(practice_messages)}개 → 하이브리드: {len(result)}개")
for msg in result:
    print(f"  [{type(msg).__name__:>13}] {msg.content[:80]}")

print("TODO를 완성해주세요")

### 실습 7-6: 전략별 토큰 사용량 비교

동일한 대화에 세 가지 전략을 적용하고 토큰 사용량을 비교하세요.

**요구사항**
1. `practice_messages`에 Sliding Window(4), Token 트리밍(100), 하이브리드 적용
2. 각 전략의 메시지 수와 토큰 수를 계산
3. 표 형태로 비교 출력

In [ ]:
# TODO: 전략별 토큰 비교

# ---------- 정답 ----------
def count_tokens(msgs):
    text = "\n".join(m.content for m in msgs)
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

# 각 전략 적용
original = practice_messages
sw = my_sliding_window(practice_messages, 4)
tok = trim_messages(practice_messages, max_tokens=100, token_counter=llm,
                     strategy="last", include_system=True, start_on="human")
hyb = my_hybrid(practice_messages, llm)

strategies = [
    ("원본", original),
    ("Sliding Window", sw),
    ("Token 트리밍", tok),
    ("하이브리드", hyb),
]

orig_tokens = count_tokens(original)
print(f"{'전략':>15} {'메시지':>6} {'토큰':>6} {'절약률':>8}")
print("-" * 40)
for name, msgs in strategies:
    t = count_tokens(msgs)
    saving = f"{(1-t/orig_tokens)*100:.1f}%" if name != "원본" else "—"
    print(f"{name:>15} {len(msgs):>6} {t:>6} {saving:>8}")

print("TODO를 완성해주세요")

### 실습 7-7: LangGraph 노드에서 트리밍

LangGraph 그래프의 노드 안에서 `trim_messages`를 사용하여 컨텍스트를 관리하세요.

**요구사항**
1. `MessagesState` 기반 StateGraph 구성
2. chatbot 노드에서 `trim_messages(max_tokens=200, token_counter=llm)` 적용
3. 3턴 대화를 실행하여 결과 확인

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

# TODO: LangGraph 노드에서 trim_messages 적용

# ---------- 정답 ----------
def my_chatbot(state: MessagesState):
    trimmed = trim_messages(
        state["messages"],
        max_tokens=200,
        token_counter=llm,
        strategy="last",
        include_system=True,
        start_on="human",
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

g = StateGraph(MessagesState)
g.add_node("chatbot", my_chatbot)

g.add_edge(START, "chatbot")
g.add_edge("chatbot", END)
my_app = g.compile(checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "practice-7-7"}}
for user_msg in ["안녕, 나는 하은이야", "파이썬 알려줘", "내 이름 기억해?"]:
    result = my_app.invoke({"messages": [HumanMessage(user_msg)]}, config=cfg)
    ai = result["messages"][-1]
    print(f"User: {user_msg}")
    print(f"AI: {ai.content[:80]}\n")

print("TODO를 완성해주세요")

User: 안녕, 나는 하은이야
AI: 안녕하세요, 하은님! 만나서 반가워요.
무엇을 도와드릴까요?

User: 파이썬 알려줘
AI: 안녕하세요, 하은님! 파이썬에 대해 궁금하시군요. 정말 좋은 선택이세요! 파이썬은 배우기 쉽고 활용 분야가 넓어 초보자부터 전문가까지 많은 사랑

User: 내 이름 기억해?
AI: 저는 인공지능이기 때문에 **개인적인 정보를 기억하거나 저장하지 않습니다.** 따라서 **사용자님의 이름을 따로 기억하고 있지는 않아요.**



TODO를 완성해주세요


---

### 생각해보기

1. 실습 7-6에서 가장 토큰 절약률이 높은 전략과 가장 정보 보존이 좋은 전략은 각각 무엇이었나요? 둘이 같은 전략인가요?
2. `trim_messages`의 `strategy="first"`는 어떤 상황에서 유용할까요? 처음 메시지를 유지하는 것이 의미 있는 시나리오를 생각해보세요.
3. LangGraph에서 `trim_messages`를 노드 안에서 적용하면, 상태에는 전체 이력이 남아 있습니다. 이것의 장점은 무엇일까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 7-1: 고객 상담 챗봇 최적 컨텍스트 전략 설계 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

온라인 쇼핑몰의 고객 상담 챗봇을 위한 최적 컨텍스트 관리 전략을 설계하세요.

**상황 설정**
- 고객이 주문, 반품, 교환 등을 문의
- 평균 대화 길이: 10~30턴
- 중요 정보: 고객 이름, 주문번호, 문의 내용, 합의사항
- 목표: 토큰 비용 절감 + 중요 정보 보존

**과제**
- 10턴 이상의 상담 시나리오 대화를 작성
- 3가지 이상의 전략을 적용하고 비교
- 각 전략으로 마지막 질문("주문번호와 합의사항을 알려주세요")에 답변시키기
- 가장 적합한 전략을 선택하고 그 이유를 설명

**평가 기준**
- 토큰 절약률
- 고객 이름/주문번호 기억 여부
- 합의사항 보존 여부

In [ ]:
# === 챌린지 7-1 답안 ===
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

# ── 1단계: 10턴 상담 시나리오 작성 ──
scenario = [
    SystemMessage("당신은 온라인 쇼핑몰 고객 상담 챗봇입니다. 정확하고 친절하게 답변하세요."),
    HumanMessage("안녕하세요, 김서연입니다. 주문 관련 문의드립니다."),
    AIMessage("안녕하세요 김서연님! 주문번호를 알려주시겠어요?"),
    HumanMessage("주문번호는 ORD-2025-88421입니다."),
    AIMessage("ORD-2025-88421 확인했습니다. 어떤 문의사항이 있으신가요?"),
    HumanMessage("지난주에 주문한 노트북이 배송이 안 왔어요."),
    AIMessage("확인해보니 배송 지연 상태입니다. 죄송합니다. 예상 도착일은 2월 25일입니다."),
    HumanMessage("너무 늦는데, 취소하고 싶어요."),
    AIMessage("취소 접수 도와드리겠습니다. 다만 이미 출고된 상태라 반품 절차로 진행됩니다."),
    HumanMessage("반품이요? 그러면 환불은 언제 되나요?"),
    AIMessage("반품 수거 후 3영업일 이내에 전액 환불됩니다. 수거는 내일 방문 예정입니다."),
    HumanMessage("알겠어요, 반품으로 진행해주세요."),
    AIMessage("반품 접수 완료했습니다. 내일 수거 방문, 수거 후 3영업일 이내 환불 처리됩니다."),
    HumanMessage("혹시 다른 모델로 교환도 가능한가요?"),
    AIMessage("네, 동일 가격 이하 모델은 교환 가능합니다. 차액은 환불됩니다."),
    HumanMessage("그러면 교환 말고 그냥 환불로 할게요."),
    AIMessage("네, 환불로 최종 확정하겠습니다."),
    HumanMessage("제 주문번호랑 최종 합의사항을 정리해주세요."),
]


# ── 2단계: 3가지 전략 적용 + 비교 ──
# 공통 토큰 계산 함수
def count_tokens(msgs):
    text = "\n".join(m.content for m in msgs)
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

# 전략 1: Sliding Window (최근 6개)
def apply_sliding_window(messages, window_size=6):
    system = [messages[0]]
    rest = messages[1:]
    return system + rest[-window_size:]

# 전략 2: Token 트리밍
def apply_token_trim(messages, max_tokens=200):
    return trim_messages(
        messages, max_tokens=max_tokens, token_counter=llm,
        strategy="last", include_system=True, start_on="human",
    )

# 전략 3: 하이브리드 (요약 + 최근 6개)
def apply_hybrid(messages, llm, keep_recent=6):
    system_msg = messages[0]
    non_system = messages[1:]
    old = non_system[:-keep_recent]
    recent = non_system[-keep_recent:]

    text = "\n".join(f"{type(m).__name__}: {m.content}" for m in old)
    summary_prompt = (
        "다음 고객 상담 대화를 요약하세요. "
        "고객 이름, 주문번호, 문의 내용, 합의사항을 반드시 포함하세요.\n\n"
        f"{text}"
    )
    summary = llm.invoke(summary_prompt).content
    new_system = SystemMessage(
        f"{system_msg.content}\n\n[이전 상담 요약]\n{summary}"
    )
    return [new_system] + recent

# 적용
strategies = {
    "원본": scenario,
    "Sliding Window": apply_sliding_window(scenario),
    "Token 트리밍": apply_token_trim(scenario),
    "하이브리드": apply_hybrid(scenario, llm),
}

orig_tokens = count_tokens(scenario)
print(f"{'전략':>15} {'메시지':>6} {'토큰':>6} {'절약률':>8}")
print("-" * 42)
for name, msgs in strategies.items():
    t = count_tokens(msgs)
    saving = f"{(1 - t / orig_tokens) * 100:.1f}%" if name != "원본" else "—"
    print(f"{name:>15} {len(msgs):>6} {t:>6} {saving:>8}")


# ── 3단계: 마지막 질문 응답 품질 비교 ──
print("\n" + "=" * 50)
print("응답 품질 비교: '주문번호와 합의사항을 정리해주세요'")
print("=" * 50)

for name, msgs in strategies.items():
    resp = llm.invoke(msgs)
    has_name = "김서연" in resp.content
    has_order = "ORD-2025-88421" in resp.content
    has_refund = "환불" in resp.content
    print(f"\n[{name}]")
    print(f"  이름: {'O' if has_name else 'X'} | 주문번호: {'O' if has_order else 'X'} | 환불 합의: {'O' if has_refund else 'X'}")
    print(f"  응답: {resp.content[:120]}")


# ── 결론 ──
print("\n" + "=" * 50)
print("결론: 고객 상담에는 하이브리드 전략이 최적")
print("=" * 50)
print("- Sliding Window/Token 트리밍: 토큰 절약률 높지만 주문번호·고객명 유실 위험")
print("- 하이브리드: 요약에 핵심 정보(이름, 주문번호, 합의사항)를 보존하면서 비용 절감")
print("- 요약 프롬프트에 '보존할 정보'를 명시하는 것이 품질의 핵심")

---

### 생각해보기

1. 고객 상담에서 "주문번호"는 절대 잊어서는 안 되는 정보입니다. 요약 기반 전략에서 이런 핵심 정보가 누락되지 않도록 어떤 장치를 추가할 수 있을까요?
2. 대화가 100턴을 넘어가면 요약 자체도 길어집니다. "요약의 요약"이 필요한 시점은 언제일까요?
3. 실제 서비스에서는 컨텍스트 관리 전략을 사용자에게 투명하게 공개해야 할까요? "이전 대화를 요약했습니다"라고 알려주는 것이 좋을까요?